# 扩展主题：检索增强生成（RAG）

> **性质**：🏗️ 工程化  ｜  **依赖**：独立 demo，不强依赖主线

## 一句话

LLM 的知识有截止日期且易幻觉。RAG 在生成前先**检索外部知识库**，把相关文档拼进 prompt，让回答有据可依——「给 LLM 开卷考试」。

## 为什么需要 RAG

| 问题 | RAG 怎么解决 |
|------|------------|
| 知识过时 | 检索最新文档，不用重训模型 |
| 容易幻觉 | 答案基于检索到的真实文档 |
| 领域知识不足 | 外挂专业文档库 |
| 无法溯源 | 可返回引用来源 |

> 对比微调：RAG 适合「知识频繁更新」的场景；微调适合「能力/风格」内化。

## RAG 三步流程

1. **索引**：把文档库切块 → 每块算 embedding → 存入向量库
2. **检索**：query 算 embedding → 与库中向量算相似度 → 取 top-k
3. **生成**：把检索到的文档拼进 prompt → 交给 LLM 生成回答

> 本 demo 用 hash 模拟 embedding（避免 sentence-transformers 依赖）。真实工程用专门的 embedding 模型 + 向量数据库。

In [ ]:
import torch
import torch.nn.functional as F

# 1. 文档库（假装是知识库）
DOCUMENTS = [
    "GPT 是一种基于 Transformer 的自回归语言模型，由 OpenAI 提出",
    "注意力机制让模型动态关注输入序列的不同部分，是 Transformer 的核心",
    "预训练是在大规模无标注文本上训练，学习语言的通用表示",
    "微调是在预训练模型基础上，用有标注数据继续训练以适应特定任务",
    "BERT 使用双向 Transformer 编码器，适合理解类任务如分类",
    "LoRA 通过低秩矩阵适配实现参数高效微调，只训练不到 1% 的参数",
    "量化把模型权重从 fp32 压缩到 int8/int4，大幅减少显存占用",
    "RLHF 通过人类反馈的强化学习，让模型输出更符合人类偏好",
]

# 2. embedding 函数（用确定性 hash 模拟，真实场景用 sentence-transformers）
EMB_DIM = 64
def embed(text, dim=EMB_DIM):
    """把文本转成向量（demo 用 hash，真实用神经网络 embedding）。"""
    torch.manual_seed(hash(text) % (2**31))
    return torch.randn(dim)

# 3. 索引：把所有文档 embed
doc_embeddings = torch.stack([embed(d) for d in DOCUMENTS])
print(f"文档库: {len(DOCUMENTS)} 篇，每篇 embedding 维度 {EMB_DIM}")
print(f"向量库形状: {tuple(doc_embeddings.shape)}")

In [ ]:
# 4. 检索函数：query embed → 余弦相似度 → top-k
def retrieve(query, k=2):
    """检索与 query 最相关的 k 篇文档。"""
    q_emb = embed(query)
    # 余弦相似度（归一化后点积）
    sims = F.cosine_similarity(q_emb.unsqueeze(0), doc_embeddings)
    topk_scores, topk_idx = sims.topk(k)
    return [(DOCUMENTS[i], topk_scores[n].item())
            for n, i in enumerate(topk_idx)]

# 测试检索
print("检索测试（hash embedding 相似度仅供参考）：")
for query in ["什么是注意力机制", "怎么省显存", "LoRA 是什么"]:
    print(f"\n查询: {query!r}")
    for doc, score in retrieve(query, k=2):
        print(f"  [{score:+.3f}] {doc[:30]}...")

## 3. 端到端 RAG：检索 + 生成

In [ ]:
def rag_answer(query, k=2):
    """完整 RAG 流程：检索相关文档 → 拼进 prompt → 生成回答。"""
    # 1. 检索
    retrieved = retrieve(query, k=k)
    # 2. 拼 prompt（真实场景喂给 LLM 生成）
    context = "\n".join(f"[{i+1}] {doc}" for i, (doc, _) in enumerate(retrieved))
    prompt = (
        f"根据以下参考资料回答问题。\n\n"
        f"参考资料：\n{context}\n\n"
        f"问题：{query}\n"
        f"回答："
    )
    return prompt, retrieved

# demo
query = "怎么让微调更省参数"
prompt, sources = rag_answer(query, k=2)
print(f"问题: {query}")
print(f"\n检索到的文档（作为回答依据）：")
for i, (doc, score) in enumerate(sources):
    print(f"  [{i+1}] (相似度 {score:+.3f}) {doc}")
print(f"\n拼成的 prompt（交给 LLM 生成）：")
print("-" * 50)
print(prompt)
print("💡 真实场景：这个 prompt 喂给 ch07 微调后的 GPT，生成有据可依的回答。")

---
> **小结**：RAG = 检索 + 生成，给 LLM 外挂可更新的知识库，解决幻觉和知识过时。
> **真实工程栈**：`sentence-transformers`(embedding) + `Chroma`/`FAISS`(向量库) + `LangChain`(编排)。
> **进阶**：重排序(rerank)、多路召回(hybrid search)、chunk 切分策略等。